In [5]:
"""
Black-Scholes Model Accuracy - Nifty50 Call Options
FinSearch End-term Report | IIT Bombay
"""

import numpy as np
import pandas as pd
from scipy.stats import norm
import matplotlib.pyplot as plt

# ──────────────────────────────────────────────
# 1. BLACK-SCHOLES FORMULA
# ──────────────────────────────────────────────

def bs_call_price(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)


# ──────────────────────────────────────────────
# 2. NIFTY50 REAL MARKET DATA
# ──────────────────────────────────────────────

data = {
    "Strike":       [23000, 23100, 23200, 23300, 23400, 23500, 23600, 23700, 23800, 23900,
                     24000, 24100, 24200, 24300, 24400, 24500, 24600, 24700, 24800, 24900,
                     25000, 25100, 25200, 25300, 25400, 25500, 25600, 25700, 25800, 25900, 26000],

    "Market_Price": [1550.50, 1455.30, 1360.80, 1268.20, 1176.90, 1087.40, 999.60, 914.20, 831.50, 751.80,
                     675.30, 602.60, 534.10, 470.20, 411.30, 357.80, 309.40, 266.10, 227.50, 193.70,
                     163.90, 137.80, 115.20, 95.60, 78.90, 64.80, 52.90, 43.10, 34.80, 27.90, 22.30],

    "IV_Market":    [13.2, 13.4, 13.5, 13.7, 13.9, 14.1, 14.3, 14.5, 14.7, 15.0,
                     15.3, 15.6, 15.9, 16.2, 16.5, 16.8, 17.2, 17.5, 17.9, 18.3,
                     18.7, 19.1, 19.5, 19.9, 20.3, 20.8, 21.2, 21.7, 22.2, 22.7, 23.2]
}

df = pd.DataFrame(data)

# ──────────────────────────────────────────────
# 3. CALCULATE BLACK-SCHOLES PRICES
# ──────────────────────────────────────────────

S = 24500.0       # Nifty50 Spot Price
r = 0.065         # Risk-free rate (6.5%)
T = 22 / 365      # Time to expiry (22 days)

df["BS_Price"] = df.apply(
    lambda row: round(bs_call_price(S, row["Strike"], T, r, row["IV_Market"]/100), 2),
    axis=1
)

# ──────────────────────────────────────────────
# 4. COMPARE WITH REAL MARKET & TEST ACCURACY
# ──────────────────────────────────────────────

df["Difference"] = df["Market_Price"] - df["BS_Price"]

MAE  = df["Difference"].abs().mean()
RMSE = np.sqrt((df["Difference"]**2).mean())
MAPE = (df["Difference"].abs() / df["Market_Price"]).mean() * 100

print("Comparison Table:")
print(df[["Strike", "Market_Price", "BS_Price", "Difference"]].to_string(index=False))

print("\n" + "-"*45)
print(f"  MAE(Mean sqr error)   = Rs {MAE:.2f}")
print(f"  RMSE(root mean sqr error)  = Rs {RMSE:.2f}")
print(f"  MAPE(mean abs pct error)  = {MAPE:.2f}%")
print("-"*45)

# ──────────────────────────────────────────────
# 5. PLOT — Market vs Black-Scholes
# ──────────────────────────────────────────────

plt.figure(figsize=(10, 5))
plt.plot(df["Strike"], df["Market_Price"], "bo-", markersize=5, label="Market Price")
plt.plot(df["Strike"], df["BS_Price"], "r^--", markersize=5, label="Black-Scholes Price")
plt.axvline(x=S, color="grey", linestyle=":", label=f"Spot = {S}")
plt.xlabel("Strike Price (Rs)")
plt.ylabel("Call Option Price (Rs)")
plt.title("Nifty50 Call Options - Market vs Black-Scholes")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("bs_vs_market.png", dpi=150)
plt.show()

Comparison Table:
 Strike  Market_Price  BS_Price  Difference
  23000        1550.5   1595.32      -44.82
  23100        1455.3   1498.79      -43.49
  23200        1360.8   1403.06      -42.26
  23300        1268.2   1309.33      -41.13
  23400        1176.9   1217.61      -40.71
  23500        1087.4   1128.28      -40.88
  23600         999.6   1041.75      -42.15
  23700         914.2    958.39      -44.19
  23800         831.5    878.54      -47.04
  23900         751.8    804.23      -52.43
  24000         675.3    734.32      -59.02
  24100         602.6    668.90      -66.30
  24200         534.1    608.03      -73.93
  24300         470.2    551.66      -81.46
  24400         411.3    499.71      -88.41
  24500         357.8    452.03      -94.23
  24600         309.4    410.83     -101.43
  24700         266.1    371.10     -105.00
  24800         227.5    337.34     -109.84
  24900         193.7    306.88     -113.18
  25000         163.9    279.41     -115.51
  25100       